# Decision Tree Training & Model Export

*Generated 2025-07-25*

This notebook trains the interpretable decision tree model from the manuscript:
1. Load de-identified routine lab data
2. Train/test split (80/20)
3. Standardize features
4. LASSO feature selection (optional if already decided)
5. Grid search for tree hyperparameters
6. Save `model.pkl`, `standard_scaler.pkl`, `feature_order.json`

Replace file paths and column names to match your environment.

## 0. Environment

In [ ]:
!pip install scikit-learn==1.6 shap==0.45 pandas numpy matplotlib seaborn --quiet

## 1. Load data
Set the CSV path and column names.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, f1_score, classification_report
import joblib, json, os

# ==== USER SETTINGS ====
CSV_PATH = 'data/your_real_data.csv'          # replace
TARGET_COL = 'NPB300'                          # 1/0 label
ID_COL = None                                  # e.g. 'SQ'
DROP_COLS = ['NTproBNP','RBC','Hb','Hct','MCH']  # remove these if present
SEED = 1

# ==== LOAD ====
df = pd.read_csv(CSV_PATH)
if ID_COL and ID_COL in df.columns:
    df_id = df[ID_COL]
else:
    df_id = None

y = df[TARGET_COL].astype(int)
X = df.drop(columns=[c for c in [TARGET_COL]+DROP_COLS if c in df.columns])
print('Shape:', X.shape)
X.head()

## 2. Train/Test split & scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print('Train:', X_train.shape, ' Test:', X_test.shape)

## 3. Optional: LASSO for feature selection

In [ ]:
# Skip if feature set is already fixed.
from sklearn.linear_model import LogisticRegression
logit = LogisticRegression(penalty='l1', solver='liblinear', max_iter=2000, random_state=SEED)
logit.fit(X_train_s, y_train)
coef = pd.Series(logit.coef_[0], index=X.columns)
keep = coef.abs() >= 1e-7
selected_features = coef.index[keep].tolist()
print('Selected:', len(selected_features))
X_train_sel = X_train[selected_features]
X_test_sel  = X_test[selected_features]

# Refit scaler
scaler = StandardScaler().fit(X_train_sel)
X_train_s = scaler.transform(X_train_sel)
X_test_s  = scaler.transform(X_test_sel)

## 4. Decision tree grid search

In [ ]:
params = {
    'max_depth': [3,4,5,6],
    'min_samples_leaf': [10,20,50],
    'min_samples_split': [20,40,80]
}
clf = DecisionTreeClassifier(random_state=SEED)
gs = GridSearchCV(clf, params, scoring='f1', cv=10, n_jobs=-1)
gs.fit(X_train_s, y_train)
print(gs.best_params_)
model = gs.best_estimator_

# Evaluate
proba = model.predict_proba(X_test_s)[:,1]
pred  = (proba >= 0.5).astype(int)
print('AUROC:', roc_auc_score(y_test, proba))
print(classification_report(y_test, pred))

## 5. Save artifacts

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(model, 'models/model.pkl')
joblib.dump(scaler,'models/standard_scaler.pkl')

feature_order = X_train_sel.columns.tolist()
with open('models/feature_order.json','w') as f:
    json.dump(feature_order, f, indent=2)
print('Saved to models/')

## 6. (Optional) Visualize tree

In [ ]:
from sklearn import tree
import matplotlib.pyplot as plt
plt.figure(figsize=(12,6))
_ = tree.plot_tree(model, feature_names=feature_order, class_names=['<=300','>300'], filled=True, rounded=True)
plt.tight_layout()
plt.savefig('docs/decision_tree.png', dpi=300)
plt.show()